In [4]:
import sys
from pathlib import Path

from fastapi import params

# Ensure project root and src/ are on sys.path so `core`, `db`, and `service` import cleanly
PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
for _p in (SRC_DIR, PROJECT_ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from gosmart_graphite.core.config import Settings
from gosmart_graphite.db.neo4j import Neo4jClient
from gosmart_graphite.service.fhir_service import FHIRBatchFetcher, FhirService
from gosmart_graphite.service.graph_service import GraphService

env_file = PROJECT_ROOT / ".env"
settings = Settings(_env_file=str(env_file) if env_file.exists() else None)

neo4j_client = Neo4jClient(settings=settings)
graph_service = GraphService(client=neo4j_client)
fhir_fetcher = FHIRBatchFetcher(base_url=settings.FHIR_BASE_URL)
fhir_service = FhirService(fetcher=fhir_fetcher, client=neo4j_client, graph_svc=graph_service)

print(f"Neo4j URI     : {neo4j_client.settings.DB_CONNECTION_URI}")
print(f"Neo4j DB      : {neo4j_client.settings.DB_DATABASE}")
print(f"FHIR Base URL : {fhir_service.fetcher.base_url}")

Neo4j URI     : neo4j+s://e2d8e1dd.databases.neo4j.io
Neo4j DB      : e2d8e1dd
FHIR Base URL : http://localhost:8088/fhir/r4


In [5]:
# Get Sample List of ImagingStudies From FHIR
import random
im_list = await fhir_fetcher.fetch_resource_bundle(
    path="ImagingStudy", params={"_count": "1000"})
if not im_list:
    raise ValueError("No ImagingStudy resources found in the FHIR server")

random_imaging_study = random.choice(im_list)
print(random_imaging_study)
imaging_study_id = random_imaging_study["id"]

{'resourceType': 'ImagingStudy', 'id': '180038', 'meta': {'versionId': '1', 'lastUpdated': '2026-10-21T13:07:48.698+00:00'}, 'identifier': [{'use': 'official', 'system': 'urn:ietf:rfc:3986', 'value': 'urn:oid:1.2.840.99999999.79710681.1716356563411'}], 'status': 'available', 'subject': {'reference': 'Patient/178742'}, 'encounter': {'reference': 'Encounter/180016'}, 'started': '2024-05-21T22:42:43-07:00', 'numberOfSeries': 1, 'numberOfInstances': 2, 'procedureCode': [{'coding': [{'system': 'http://snomed.info/sct', 'code': '314971001', 'display': 'Camera fundoscopy (procedure)'}], 'text': 'Camera fundoscopy (procedure)'}], 'location': {'reference': 'Location/1769', 'display': 'ASSOCIATED PHYSICIANS OF HARVARD MEDICAL FACULTY PHYSICIANS AT BETH IS'}, 'series': [{'uid': '1.2.840.99999999.1.89931948.1716356563411', 'number': 1, 'modality': {'system': 'http://dicom.nema.org/medical/dicom/current/output/chtml/part16/sect_CID_29.html', 'code': 'OP', 'display': 'Ophthalmic Photography'}, 'numb

# Configurations

# Traversal and Integrity Check from ImagingStudy Resource

1. Get the ImagingStudy resource
2. Get the Encounter resource pointed to by the ImagingStudy resource
3. Get all other ImaginStudy resources that link to the Encounter
4. Get the ImagingStudy resources directly from FHIR to ensure that they also refer to the same encounter

In [6]:
from fhir.resources.R4B.imagingstudy import ImagingStudy

study_lookup_id = str(imaging_study_id).strip()
print(f"=== Traversal & Integrity Check for ImagingStudy: {study_lookup_id} ===\n")

# 1. Get the ImagingStudy resource from Neo4j
img_records = await neo4j_client.execute_query(
    """
    MATCH (img:ImagingStudy)
    WHERE (img.resource_id = $lookup_id OR img.id = $lookup_id OR img.studyUid = $lookup_id)
      AND (img.Action IS NULL OR img.Action <> 'DELETE')
    RETURN properties(img) AS img
    """,
    {"lookup_id": study_lookup_id},
)

if not img_records:
    sample_records = await neo4j_client.execute_query(
        """
        MATCH (img:ImagingStudy)-[:PERFORMED_IN]->(e:Encounter)<-[:PERFORMED_IN]-(other:ImagingStudy)
        WHERE img.id <> other.id
          AND (img.Action IS NULL OR img.Action <> 'DELETE')
        RETURN DISTINCT img.resource_id AS resource_id, img.studyUid AS studyUid, e.resource_id AS encounter_id
        LIMIT 5
        """
    )
    sample_ids = [r["resource_id"] for r in sample_records]
    raise ValueError(
        f"ImagingStudy '{study_lookup_id}' not found in Neo4j. "
        f"Sample valid ImagingStudy resource_ids with shared encounters: {sample_ids}"
    )

img_node = img_records[0]["img"]
img_res_id = img_node["resource_id"]
print("[1] Target ImagingStudy Node (Neo4j):")
print(f"    node id           : {img_node.get('id')}")
print(f"    resource_id       : {img_res_id}")
print(f"    studyUid          : {img_node.get('studyUid')}")
print(f"    procedureCodeText : {img_node.get('procedureCodeText')}")
print(f"    encounter prop    : {img_node.get('encounter')}\n")

# 2. Get the Encounter resource pointed to by the ImagingStudy resource
enc_records = await neo4j_client.execute_query(
    """
    MATCH (img:ImagingStudy {id: $img_node_id})-[:PERFORMED_IN]->(enc:Encounter)
    WHERE (enc.Action IS NULL OR enc.Action <> 'DELETE')
    RETURN properties(enc) AS enc
    """,
    {"img_node_id": img_node["id"]},
)

if not enc_records:
    raise ValueError(f"No Encounter linked via [:PERFORMED_IN] from ImagingStudy '{img_res_id}'.")

enc_node = enc_records[0]["enc"]
enc_res_id = enc_node["resource_id"]
expected_enc_ref = f"Encounter/{enc_res_id}"
print("[2] Linked Encounter Node (Neo4j):")
print(f"    node id           : {enc_node.get('id')}")
print(f"    resource_id       : {enc_res_id}")
print(f"    status            : {enc_node.get('status')}")
print(f"    class             : {enc_node.get('class')}")
print(f"    periodStart       : {enc_node.get('periodStart')}\n")

# 3. Get all other ImagingStudy resources that link to the Encounter
other_img_records = await neo4j_client.execute_query(
    """
    MATCH (enc:Encounter {id: $enc_node_id})<-[:PERFORMED_IN]-(other:ImagingStudy)
    WHERE other.id <> $img_node_id
      AND (other.Action IS NULL OR other.Action <> 'DELETE')
    RETURN properties(other) AS other
    ORDER BY other.resource_id
    """,
    {"enc_node_id": enc_node["id"], "img_node_id": img_node["id"]},
)

other_img_nodes = [r["other"] for r in other_img_records]
print(f"[3] Other ImagingStudy Nodes Linked to {expected_enc_ref} in Neo4j ({len(other_img_nodes)} found):")
for other in other_img_nodes:
    print(
        f"    - ImagingStudy/{other.get('resource_id')} "
        f"(studyUid={other.get('studyUid')}, procedure={other.get('procedureCodeText')}, "
        f"encounter={other.get('encounter')})"
    )
print()

# 4. Get the ImagingStudy resources directly from FHIR and verify they refer to the same Encounter
all_graph_img_nodes = [img_node, *other_img_nodes]
print(f"[4] Direct FHIR Verification ({len(all_graph_img_nodes)} ImagingStudy resource(s)):")

mismatches = []
for node in all_graph_img_nodes:
    res_id = node["resource_id"]
    raw_fhir = await fhir_service.fetcher.fetch_resource("ImagingStudy", res_id)
    if raw_fhir is None:
        mismatches.append((res_id, "Missing in FHIR", expected_enc_ref))
        print(f"    [FAIL] ImagingStudy/{res_id}: not found on FHIR server")
        continue

    fhir_img = ImagingStudy.model_validate(raw_fhir)
    fhir_enc_ref = fhir_img.encounter.reference if fhir_img.encounter else None
    normalized_fhir_enc = (
        f"Encounter/{fhir_enc_ref.strip('/').split('/')[-1]}" if fhir_enc_ref else None
    )
    matches = normalized_fhir_enc == expected_enc_ref
    status_label = "OK" if matches else "MISMATCH"
    print(
        f"    [{status_label}] ImagingStudy/{res_id}: "
        f"FHIR encounter.reference='{fhir_enc_ref}' (expected '{expected_enc_ref}')"
    )
    if not matches:
        mismatches.append((res_id, fhir_enc_ref, expected_enc_ref))

# Also verify FHIR search bundle for ImagingStudy?encounter=<enc_res_id> matches the Neo4j set
fhir_bundle_studies = await fhir_service.fetcher.fetch_resource_bundle(
    "ImagingStudy", {"encounter": enc_res_id}
)
fhir_bundle_ids = sorted(str(r.get("id")) for r in fhir_bundle_studies if r.get("id"))
graph_ids = sorted(str(n["resource_id"]) for n in all_graph_img_nodes)
print(f"\n    Graph ImagingStudy IDs for {expected_enc_ref} : {graph_ids}")
print(f"    FHIR  ImagingStudy IDs for {expected_enc_ref} : {fhir_bundle_ids}")

assert not mismatches, f"Encounter reference mismatches detected: {mismatches}"
assert graph_ids == fhir_bundle_ids, (
    f"Set mismatch between Neo4j ({graph_ids}) and FHIR ({fhir_bundle_ids})"
)
print("\nINTEGRITY CHECK PASSED: All ImagingStudy resources in Neo4j and FHIR refer to the exact same Encounter.")

Unable to retrieve routing information


=== Traversal & Integrity Check for ImagingStudy: 180038 ===



ServiceUnavailable: Unable to retrieve routing information